# EXP-03 — HDBSCAN Adaptive & Multi-Sequence Resolution

Evaluates HDBSCAN with adaptive parameters and multi-sequence entity resolution on reproduction dataset:
1. **Deterministic Adaptive Parameters**: `min_cluster_size` and `min_samples` scaled from candidate count $N$ and mainshock magnitude $M$ without per-sequence manual overrides.
2. **Raw Membership Grain Preservation**: Retains every candidate-mainshock pair `(experiment_run_id, mainshock_event_id, candidate_event_id)`.
3. **Multi-Sequence Entity Resolution**: Deterministic tie-breaking hierarchy: `membership_probability` (higher wins) → `normalized_distance` (lower wins) → `AMBIGUOUS` (unresolvable tie, no arbitrary selection).
4. **Nested Mainshock Detection**: Mainshocks identified as sequence members in other windows are linked via `is_nested_mainshock_candidate`, `parent_mainshock_event_id`, and `nested_membership_probability`.
5. **Before/After Reconciliation**: Complete deduplication audit demonstrating zero double counting on resolved rank-1 events.

In [ ]:
import json
import os
import sys
import tempfile
from pathlib import Path

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'notebooks/adaptive_hdbscan.py').is_file(), 'Run notebook from repository root or notebooks/ directory'
sys.path.insert(0, str(root / 'notebooks'))
sys.path.insert(0, str(root / 'tests'))

from experiment_harness import load_feature_bundle, MODE_DEVELOPMENT, VECTOR_COLUMNS
from adaptive_hdbscan import (
    resolve_adaptive_hdbscan_params,
    fit_hdbscan_adaptive,
    resolve_multi_sequence_and_nested_mainshocks,
    run_adaptive_experiment,
)
print('EXP-03 Adaptive HDBSCAN modules imported successfully.')

## 1. Load Feature Bundle

Loads input feature dataset and verifies checksums, schema, and non-finite constraints.

In [ ]:
from test_experiment_harness import create_pilot_fixture

bundle_path = os.environ.get('PILOT_BUNDLE', '')
_temp_dir = None
if bundle_path and Path(bundle_path).is_dir():
    pilot_dir = Path(bundle_path).resolve()
else:
    _temp_dir = tempfile.TemporaryDirectory()
    pilot_dir = Path(_temp_dir.name)
    create_pilot_fixture(pilot_dir)

bundle = load_feature_bundle(pilot_dir, mode=MODE_DEVELOPMENT)
assert bundle['mode'] == MODE_DEVELOPMENT
features = bundle['features_table']
print('Loaded Dataset ID:', bundle['bundle_id'])
print('Candidate count:', features.num_rows)
print('Feature columns:', VECTOR_COLUMNS)
assert features.num_rows > 0

## 2. Load Model Configuration and Demonstrate Adaptive Parameter Resolver

Evaluates the mathematical scaling rule across window sizes.

In [ ]:
cfg_path = root / 'config/ml/models/hdbscan_adaptive_dev_v1.json'
with open(cfg_path, 'r', encoding='utf-8') as f:
    model_config = json.load(f)

print('Algorithm:', model_config['algorithm_name'])
print('Model Config Version:', model_config['model_config_version'])

# Sample scaling scenarios
test_scenarios = [
    (5, 5.5, 'Small window, baseline magnitude'),
    (50, 6.0, 'Medium window, moderate magnitude'),
    (250, 6.8, 'Large sequence, strong magnitude'),
    (1500, 7.5, 'Very large sequence, major earthquake'),
]
print('\n--- Adaptive Scaling Demonstration ---')
for n, m, desc in test_scenarios:
    p = resolve_adaptive_hdbscan_params(n, m, model_config)
    print(f"{desc}: N={n}, M={m} -> min_cluster_size={p['min_cluster_size']}, min_samples={p['min_samples']}")
    assert p['min_cluster_size'] >= 3
    assert p['min_samples'] <= p['min_cluster_size']

## 3. Fit HDBSCAN Adaptive and Resolve Multi-Sequences

Runs full pipeline: window fitting -> multi-sequence entity resolution -> nested mainshock detection.

In [ ]:
_out_temp = tempfile.TemporaryDirectory()
output_dir = Path(_out_temp.name) / 'adaptive_experiment'

result = run_adaptive_experiment(
    bundle=bundle,
    config=model_config,
    model_config_version=model_config['model_config_version'],
    output_dir=output_dir,
    mode=MODE_DEVELOPMENT,
)

recon = result['reconciliation']
metrics = result['metrics']

print('=== EXP-03 Execution Summary ===')
print(f"Experiment Run ID: {result['run_id']}")
print(f"Total Windows Processed: {metrics['total_windows']}")
print(f"Total Raw Memberships: {recon['total_raw_memberships']}")
print(f"Raw Sequence Members: {recon['raw_sequence_members']}")
print(f"Raw Aftershocks: {recon['raw_aftershocks']}")
print(f"Multi-Sequence Candidates: {recon['unique_candidates_in_multi_sequences']}")
print(f"Resolved Unique Aftershocks (Rank 1): {recon['resolved_unique_aftershocks']}")
print(f"Deduplicated Duplicate Aftershocks: {recon['aftershocks_deduplicated_count']}")
print(f"Ambiguous Candidates: {recon['ambiguous_candidates_count']}")
print(f"Nested Mainshocks Detected: {recon['nested_mainshocks_count']}")
print(f"Total Runtime: {metrics['runtime_seconds']}s")

## 4. Verification and Contract Integrity

Confirms outputs, schemas, and checksums match `result-bundle-v1` specifications.

In [ ]:
import pyarrow.parquet as pq

mem_path = output_dir / 'memberships.parquet'
sum_path = output_dir / 'sequence_summary.parquet'
manifest_path = output_dir / 'pilot_result_manifest.json'

assert mem_path.is_file(), 'Missing memberships.parquet'
assert sum_path.is_file(), 'Missing sequence_summary.parquet'
assert manifest_path.is_file(), 'Missing pilot_result_manifest.json'
assert not (output_dir / '_SUCCESS.json').exists(), 'DEVELOPMENT mode must never write _SUCCESS.json'

mem_table = pq.read_table(mem_path)
sum_table = pq.read_table(sum_path)

assert mem_table.num_rows == features.num_rows, 'Membership count must match feature candidate count'
assert sum_table.num_rows == metrics['total_windows'], 'Summary count must match window count'

statuses = set(mem_table['multi_sequence_status'].to_pylist())
assert statuses.issubset({'UNIQUE', 'MULTIPLE', 'AMBIGUOUS', 'NOT_MEMBER'}), f'Invalid statuses: {statuses}'

print('EXP-03 Adaptive HDBSCAN and Entity Resolution verified successfully.')